# Installation

In [1]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch==2.7.1" "triton>=3.3.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

# These are mamba kernels and we must have these for faster training
# Mamba kernels are for now supported only on torch==2.7.1. If you have newer torch versions, please wait 30 minutes for it to compile
!uv pip install --no-build-isolation mamba_ssm==2.2.5
!uv pip install --no-build-isolation causal_conv1d==1.5.2

%%capture
import os, importlib.util
!pip install --upgrade -qqq uv

if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0 xformers==0.0.32.post2

# Unsloth

In [2]:
from unsloth import FastLanguageModel
import torch

"""
Model explanations:

granite-4.0-h-micro: 3B dense model
granite-4.0-h-tiny: hybrid 7B MoE w/ 1B active
granite-4.0-h-small: hybrid 32B MoE w/ 9B active

granite-3.3-8b-instruct: 8B dense model
"""

fourbit_models = [
    "unsloth/granite-4.0-micro",
    "unsloth/granite-4.0-h-micro",
    "unsloth/granite-4.0-h-tiny",
    "unsloth/granite-4.0-h-small",

    # Base pretrained Granite 4 models
    "unsloth/granite-4.0-micro-base",
    "unsloth/granite-4.0-h-micro-base",
    "unsloth/granite-4.0-h-tiny-base",
    "unsloth/granite-4.0-h-small-base",
]

# We can try testing granite 4 small on a big GPU if micro isn't that good
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/granite-4.0-h-tiny",
    max_seq_length = 1024,   # Same as inference script
    load_in_4bit = False,    # Load full precision for max accuracy
    load_in_8bit = False,
    full_finetuning = False, # Don't do this, LoRA gets just as good results
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Granitemoehybrid patching. Transformers: 4.56.2.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.0. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00003.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00002-of-00003.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/4.07G [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

We now add LoRA adapters so we only need to update a small amount of parameters!

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Usually multiples of 8
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",
                      "shared_mlp.input_linear", "shared_mlp.output_linear"],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # Just go with Unsloth recommendation to set alpha = rank
    loftq_config = None, # Not quantizing so don't need this
)

Unsloth: Detected MoE model with num_experts = 64 and target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj', 'shared_mlp.input_linear', 'shared_mlp.output_linear']. Enabling LoRA on MoE parameters: ['mlp.experts.gate_up_proj', 'mlp.experts.down_proj']
Unsloth: PEFT set target_parameters but found no matching parameters.
This is expected for MoE models - Unsloth handles MoE expert LoRA targeting separately.
Unsloth: Making `model.base_model.model.model` require gradients


# Data Prep

The chat template for granite-4 look like this:
```
<|start_of_role|>system<|end_of_role|>Knowledge Cutoff Date: April 2024.
Today's Date: June 24, 2025.
You are Granite, developed by IBM. You are a helpful AI assistant.<|end_of_text|>

<|start_of_role|>user<|end_of_role|>How do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?<|end_of_text|>

<|start_of_role|>assistant<|end_of_role|>Astronomers make use of the unique spectral fingerprints of elements found in stars...<|end_of_text|>
```

Need to format the data into the following conversational style:

```
{"role": "system", "content": "system prompt"}
{"role": "user", "content": "What is 2+2?"}
{"role": "assistant", "content": "It's 4."}
```

In [4]:
# From Sakhawat et al., 2026

default_system_prompt = """
You are participating in a standardized News Bias Classification task for academic
research. Output ONLY a single numeric value between -3.0 and +3.0.

Do NOT provide explanations or text.
"""

In [5]:
# Load the dataset
from datasets import load_dataset

train_dataset = load_dataset('avanishd/ground-news-2026', split='train')
validation_dataset = load_dataset('avanishd/ground-news-2026', split='validation')

README.md:   0%|          | 0.00/679 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/395k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/375k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8642 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1884 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1836 [00:00<?, ? examples/s]

In [6]:
def formatting_prompts_func(examples):

    bias_mapping = {
        "Far Left": "-3.0",
        "Left": "-2.0",
        "Lean Left": "-1.0",
        "Center": "0.0",
        "Lean Right": "+1.0",
        "Right": "+2.0",
        "Far Right": "+3.0"
    }

    messages = [
        [
            {"role": "system", "content": default_system_prompt},
            {"role": "user", "content": f"""
            HEADLINE: {headline}
            ARTICLE SUMMARY: {summary}

            Output ONLY the numeric bias score.
            """},
            {"role": "assistant", "content": bias_mapping[true_bias]}
        ] for headline, summary, true_bias in zip(examples['headline'], examples['summary'], examples['bias'])
    ]

    texts = [tokenizer.apply_chat_template(message, tokenize = False, add_generation_prompt = False) for message in messages]

    return { "text" : texts, }

train_dataset = train_dataset.map(formatting_prompts_func, batched = True,)
validation_dataset = validation_dataset.map(formatting_prompts_func, batched = True,)

Map:   0%|          | 0/8642 [00:00<?, ? examples/s]

Map:   0%|          | 0/1836 [00:00<?, ? examples/s]

Look at how the chat template mapped the conversation

In [7]:
train_dataset[5]["text"]

"<|start_of_role|>system<|end_of_role|>\nYou are participating in a standardized News Bias Classification task for academic\nresearch. Output ONLY a single numeric value between -3.0 and +3.0.\n\nDo NOT provide explanations or text.\n<|end_of_text|>\n<|start_of_role|>user<|end_of_role|>\n            HEADLINE: Three New Rulings, One Goes President Trump's Way\n            ARTICLE SUMMARY: Three new rulings and one unexpected victory for Donald Trump. Pursuing Chief Lyons Chief Judge Patrick Schiltz of Minnesota’s federal court ordered the acting chief of U.S. Immigration and Customs Enforcement, Todd Lyons, to appear in court on Friday. He must personally explain why that agency has not complied with a slew of court orders. […] The post Three New Rulings, One Goes President Trump’s Way appeared first on www.independentsentinel.co…\n\n            Output ONLY the numeric bias score.\n            <|end_of_text|>\n<|start_of_role|>assistant<|end_of_role|>+2.0<|end_of_text|>\n"

# Training


Effective batch size = per_device_train_batch_size * gradient_accumulation steps

With Unsloth, per_device_train_batch_size and gradient_accumulation steps are equivalent, which may not be the case in other frameworks.

This setup (with effective batch size 32) is the fastest one. Having any gradient accumulation steps slows Granite down.

In [8]:
from trl import SFTTrainer, SFTConfig

# 3 evals per epoch
PER_DEVICE_TRAIN_BATCH_SIZE = 32
GRADIENT_ACCUMULATION_STEPS = 1

EFFECTIVE_BATCH_SIZE = PER_DEVICE_TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS

EVAL_STEPS = len(train_dataset) // EFFECTIVE_BATCH_SIZE // 3

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = validation_dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE,
        gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS, # Use GA to mimic batch size!
        warmup_ratio = 0.05,
        num_train_epochs = 1, # Full training runs, can experiment w/ multiple
 #       max_steps = 60,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        eval_strategy = "steps",
        eval_steps = EVAL_STEPS,
        per_device_eval_batch_size = PER_DEVICE_TRAIN_BATCH_SIZE, # Using higher batch size to speed up eval
        eval_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc (TODO: Set up when dataset is finalized)
    ),
)



Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/8642 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1836 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [9]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|start_of_role|>user|end_of_role|>",
    response_part = "<|start_of_role|>assistant<|end_of_role|>",
)

Map (num_proc=16):   0%|          | 0/8642 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/8642 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/1836 [00:00<?, ? examples/s]

Filter (num_proc=16):   0%|          | 0/1836 [00:00<?, ? examples/s]

Verify masking the instruction part is done

In [10]:
tokenizer.decode(trainer.train_dataset[0]["input_ids"])

'<|start_of_role|>system<|end_of_role|>\nYou are participating in a standardized News Bias Classification task for academic\nresearch. Output ONLY a single numeric value between -3.0 and +3.0.\n\nDo NOT provide explanations or text.\n<|end_of_text|>\n<|start_of_role|>user<|end_of_role|>\n            HEADLINE: Trump picks Kevin Warsh for Fed chair, but key Republican vows to block him over Powell investigation\n            ARTICLE SUMMARY: President Donald Trump announced conservative economist and former Fed governor Kevin Warsh as his pick to be the new Federal Reserve chairman.\n\n            Output ONLY the numeric bias score.\n            <|end_of_text|>\n<|start_of_role|>assistant<|end_of_role|>-1.0<|end_of_text|>\n'

Print out the masked out example - should only see the assistant response

In [11]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[0]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                                                                   -1.0<|end_of_text|>\n'

In [12]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A100-SXM4-40GB. Max memory = 39.494 GB.
13.018 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

```
Notice you might have to wait ~10 minutes for the Mamba kernels to compile! Please be patient!
```

In [13]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,642 | Num Epochs = 1 | Total steps = 271
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 9,175,040 of 6,948,212,288 (0.13% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss,Validation Loss
90,0.245000,0.258874
180,0.179000,0.170649
270,0.156100,0.152844


GraniteMoeHybrid requires an initialized `HybridMambaAttentionDynamicCache` to return a cache. Because one was not provided, no cache will be returned.


In [14]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

638.9402 seconds used for training.
10.65 minutes used for training.
Peak reserved memory = 27.398 GB.
Peak reserved memory for training = 14.38 GB.
Peak reserved memory % of max memory = 69.373 %.
Peak reserved memory for training % of max memory = 36.411 %.


# Saving model

In [15]:
from google.colab import userdata
model.push_to_hub_merged("avanishd/granite-4.0-h-tiny-finetuned-ground-news", tokenizer, save_method = "merged_16bit", token = userdata.get('HF_TOKEN'))


No files have been modified since last commit. Skipping to prevent empty commit.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


No files have been modified since last commit. Skipping to prevent empty commit.


Checking cache directory for required files...


Unsloth: Copying 3 files from cache to `avanishd/granite-4.0-h-tiny-finetuned-ground-news`: 100%|██████████| 3/3 [00:35<00:00, 11.71s/it]


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...0003-of-00003.safetensors:   0%|          | 15.9MB / 4.07GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 3/3 [03:17<00:00, 65.87s/it]


Unsloth: Merge process complete. Saved to `/content/avanishd/granite-4.0-h-tiny-finetuned-ground-news`
